# Bilişsel Performans Tahmini

Herkese merhaba. Bu çalışmada, bireylerin yaşam tarzı alışkanlıklarına (uyku, stres, fiziksel aktivite vb.) dayalı olarak bilişsel performanslarını tahmin etmeyi amaçladık.

Projeye başlarken önümüzde iki seçenek vardı: Ya AutoGluon gibi utoML araçlarına veriyi verip arkama yaslanacaktık ya da aldığımız veri bilimi eğitiminin hakkını vererek verinin doğasını anlayan, istatistiksel olarak açıklanabilir ve aşırı öğrenmeye (overfitting) karşı dirençli bir mimari kuracaktık. Biz zor ama doğru olan ikinci yolu seçtik.

Bu notebook'ta adım adım; verinin anatomisini nasıl çıkardığımızı, gürültülü uç değerleri nasıl izole ettiğimizi ve davranışsal özellikleri makine öğrenmesi algoritmalarına nasıl entegre ettiğimizi göreceksiniz.

In [ ]:
pip install catboost lightgbm optuna

In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import root_mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from scipy.optimize import minimize
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from lightgbm import early_stopping as lgb_early_stopping

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_theme(style="whitegrid", palette="muted")

## 1. Keşifçi Veri Analizi (EDA):
Modellemeye geçmeden önce verinin bize anlatmak istediklerini görmek istedik. Veri setinin boyutlarına, eksik değerlerine ve temel istatistiklerine göz atarak işe başladık. Hedef değişkenimiz olan `bilissel_performans_skoru`nun dağılımı ve diğer sayısal değişkenlerle olan korelasyonu, kuracağım modelin temelini oluşturacak.

In [ ]:
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test_x.csv')
test_ids = test_df['id'].copy()
TARGET = 'bilissel_performans_skoru'

print(f"Eğitim Seti Boyutu: {train_df.shape}")
print(f"Test Seti Boyutu: {test_df.shape}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(train_df[TARGET], kde=True, color='royalblue', bins=30, ax=axes[0])
axes[0].set_title('Bilişsel Performans Skoru Dağılımı', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Skor (0-10)')
axes[0].set_ylabel('Kişi Sayısı')

numeric_df = train_df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
top_corr_features = corr.index[abs(corr[TARGET]) > 0.1]
sns.heatmap(train_df[top_corr_features].corr(), annot=True, cmap='coolwarm', fmt=".2f",
            linewidths=0.5, ax=axes[1], cbar_kws={"shrink": .8})
axes[1].set_title('En Güçlü İlişkiler (Korelasyon Matrisi)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nVeri Setinden İlk 5 Satır:")
pd.set_option('display.max_columns', None)
display(train_df.head())

print("\nEğitim Veri Seti Bilgileri:")
train_df.info()

print("\nİstatistiksel Özet:")
display(train_df.describe().T)

In [ ]:
print(" Veri Setinden İlk 5 Satır:")
pd.set_option('display.max_columns', None)
train_df.head()


### Aykırı Değerler (Outliers) ile Başa Çıkma
Veri setine baktığımda bazı yerlerde mantıksız uç değerler vardı. Örneğin kafein tüketimi, ekran süresi ve şekerleme sürelerindeki anormallikleri görmek için Kutu Grafikleri (Boxplot) oluşturduk.

Aşağıdaki tablolarda göreceğimiz bu aşırı uç değerler, bize eksik veri doldurma aşamasında KNN Imputer veya Ortalama (Mean) yerine, uç değerlere karşı çok daha dirençli olan **Medyan (Ortanca)** kullanmamız gerektiğini kanıtladı.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(x=train_df['uyku_oncesi_kafein_mg'], color='salmon', ax=axes[0])
axes[0].set_title('Kafein Tüketimi (Aykırı Değerler)', fontweight='bold')

sns.boxplot(x=train_df['uyku_oncesi_ekran_suresi_dk'], color='lightgreen', ax=axes[1])
axes[1].set_title('Ekran Süresi (Aykırı Değerler)', fontweight='bold')

sns.boxplot(x=train_df["sekerleme_suresi_dk"], color='lightblue', ax=axes[2])
axes[2].set_title('Şekerleme Süresi (Aykırı Değerler)', fontweight='bold')

plt.tight_layout()
plt.show()

Sayısal verileri inceledikten sonra meslek grupları, yaş ve uyku evrelerinin (REM) hedef değişken üzerindeki etkilerine odaklandım. Özellikle Ekonometrik nedensellik açısından, belirli bir grubun (örneğin Emeklilerin) REM uykusu ile bilişsel performansları arasındaki ilişkiyi regresyon eğrisi ile modelledim.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Meslek Tipi ve Bilişsel Performans
sns.boxplot(x='meslek', y=TARGET, data=train_df, palette='Set2', ax=axes[0])
axes[0].set_title('Meslek Tipi ve Performans İlişkisi', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Emeklilerde REM ve Performans Regresyonu
emekli_df = train_df[train_df['meslek'] == 'Emekli']
sns.scatterplot(x='rem_yuzdesi', y=TARGET, data=emekli_df, color='coral', s=80, alpha=0.7, label='Bireyler', ax=axes[1])
sns.regplot(x='rem_yuzdesi', y=TARGET, data=emekli_df, scatter=False, color='darkblue', line_kws={"linewidth":2}, label='Eğilim Çizgisi', ax=axes[1])
axes[1].set_title('Emeklilerde REM Uykusu Regresyonu', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()
print("Kategorik Değişkenler:")
print(cat_cols)

for col in cat_cols:
    print(f"\n'{col}' Değişkeninin Değer Dağılımı:")
    print(train_df[col].value_counts())

missing_values = train_df.isnull().sum()
print("\nEksik Değer Sayısı:")
print(missing_values)

## 2. Veri Ön İşleme
Keşifçi veri analizinden çıkardığım derslerle veriyi temizlemeye başladım.
1. **Ülke İsimleri:** Kategorik ayrışmayı önlemek için Türkçe-İngilizce isim karmaşasını giderdik.
2. **Eksik Veriler:** Boxplot'larda gördüğümüz şiddetli aykırı değerlerden dolayı, modeli yanıltmamak adına sayısal eksiklikleri **Medyan** ile doldurdum. Kategorik değerlerde ise en çok tekrar eden değeri (Mod) baz aldım.

In [ ]:
# Ülke İsimlerini Standartlaştırma
ulke_replace = {
    'South Korea': 'Guney Kore', 'Mexico': 'Meksika',
    'Netherlands': 'Hollanda', 'Spain': 'Ispanya', 'Sweden': 'Isvec'
}
train_df['ulke'] = train_df['ulke'].replace(ulke_replace)
test_df['ulke'] = test_df['ulke'].replace(ulke_replace)

# Eksik Değer Doldurma (Medyan ve Mod)
cat_cols_initial = train_df.select_dtypes(include=["object"]).columns.tolist()
num_cols_missing = [col for col in train_df.select_dtypes(include=[np.number]).columns if col not in ["id", TARGET]]

num_imputer = SimpleImputer(strategy="median")
train_clean = train_df.copy()
test_clean = test_df.copy()

train_clean[num_cols_missing] = num_imputer.fit_transform(train_df[num_cols_missing])
test_clean[num_cols_missing] = num_imputer.transform(test_df[num_cols_missing])

for col in cat_cols_initial:
    mode_val = train_clean[col].mode()[0]
    train_clean[col] = train_clean[col].fillna(mode_val)
    test_clean[col] = test_clean[col].fillna(mode_val)

## 3. Özellik Mühendisliği (Feature Engineering)
Burası modelimin sadece sayılara değil, insan davranışlarına da hakim olmasını sağladığımız yer. Ağaç tabanlı algoritmalar değişkenleri tek başına iyi ayırsa da, çapraz ilişkileri kurmakta zorlanıyorlardı.

Bu yüzden:
* **Etkileşim Terimleri:** Stres ve uykusuzluğun birlikte nasıl yıkıcı bir etki yarattığını modele göstermek için `stres_calisma`, `dinlenme_stres` gibi ekonometrik çarpımlar/oranlar oluşturdum.
* **Tıbbi Eşikler:** İdeal REM (%20-25) ve Derin Uyku (%13-23) dilimlerini baz alarak kişilerin uyku kalitelerini kategorize ettim.
* **Target Clipping:** Modelin hatasını (RMSE) suni olarak şişiren en alt ve en üst %0.5'lik skora sahip aykırı değerleri kestim.

In [ ]:
upper = train_clean[TARGET].quantile(0.995)
lower = train_clean[TARGET].quantile(0.005)
train_clean[TARGET] = np.clip(train_clean[TARGET], lower, upper)
print(f"Target clipped: [{lower:.3f}, {upper:.3f}]")

def feature_engineering(df):
    d = df.copy()
    # Kalite ve bileşenler
    d["uyku_kalite"] = d["rem_yuzdesi"] + d["derin_uyku_yuzdesi"]
    d["hafif_uyku"] = 100 - d["uyku_kalite"]
    d["rem_derin_carpim"] = d["rem_yuzdesi"] * d["derin_uyku_yuzdesi"]
    d["uyku_verimi"] = d["rem_derin_carpim"] / (d["gecelik_uyanma_sayisi"] + 1)
    d["uyku_baskisi"] = d["gecelik_uyanma_sayisi"] * d["uykuya_dalma_suresi_dk"]
    d["stres_calisma"] = d["stres_skoru"] * d["gunluk_calisma_saati"]
    d["rem_stres_oran"] = d["rem_yuzdesi"] / (d["stres_skoru"] + 1)
    d["dinlenme_stres"] = d["uyku_kalite"] / (d["stres_skoru"] + 1)
    d["kafein_ekran"] = (d["uyku_oncesi_kafein_mg"] / 100) + (d["uyku_oncesi_ekran_suresi_dk"] / 60)
    d["adim_calisma_oran"] = d["gunluk_adim_sayisi"] / (d["gunluk_calisma_saati"] + 1)
    #Etkileşimler
    d["stres_x_uyku"] = d["stres_skoru"] * d["uyku_kalite"]
    d["uyanma_x_rem"] = d["gecelik_uyanma_sayisi"] * d["rem_yuzdesi"]
    d["rem_x_adim"] = d["rem_yuzdesi"] * (d["gunluk_adim_sayisi"] / 1000)
    d["derin_x_stres"] = d["derin_uyku_yuzdesi"] * d["stres_skoru"]
    d["kafein_x_stres"] = d["uyku_oncesi_kafein_mg"] * d["stres_skoru"]
    d["uyku_fragmentasyonu"] = d["gecelik_uyanma_sayisi"] * d["hafif_uyku"]

    # Oranlar
    d["rem_derin_oran"] = d["rem_yuzdesi"] / (d["derin_uyku_yuzdesi"] + 1)
    d["adim_yas_oran"] = d["gunluk_adim_sayisi"] / (d["yas"] + 1)

    # Log dönüşümleri
    d["stres_log"] = np.log1p(d["stres_skoru"])
    d["kafein_log"] = np.log1p(d["uyku_oncesi_kafein_mg"])
    d["ekran_log"] = np.log1p(d["uyku_oncesi_ekran_suresi_dk"])

    # Polinom
    d["rem_karesi"] = d["rem_yuzdesi"] ** 2
    d["derin_karesi"] = d["derin_uyku_yuzdesi"] ** 2

    # Flag'ler
    d["hafta_sonu"] = (d["gun_tipi"] == "Hafta sonu").astype(int)
    d["stres_yuksek"] = (d["stres_skoru"] > 7).astype(int)
    d["ruh_sagligi_saglikli"] = (d["ruh_sagligi_durumu"] == "Saglikli").astype(int)

    # Tıbbi Eşikler (yeni kategorik değişkenler)
    d['rem_durumu'] = pd.cut(d['rem_yuzdesi'], bins=[0, 19.9, 25.1, 100],
                            labels=['Yetersiz', 'Ideal', 'Fazla']).astype(str)
    d['derin_durumu'] = pd.cut(d['derin_uyku_yuzdesi'], bins=[0, 12.9, 23.1, 100],
                            labels=['Yetersiz', 'Ideal', 'Fazla']).astype(str)
    return d

train_fe = feature_engineering(train_clean)
test_fe = feature_engineering(test_clean)

cat_cols = ["cinsiyet", "meslek", "ulke", "kronotip", "mevsim", "gun_tipi",
            "ruh_sagligi_durumu", "rem_durumu", "derin_durumu"]
for col in cat_cols:
    train_fe[col] = train_fe[col].astype(str)
    test_fe[col] = test_fe[col].astype(str)

X = train_fe.drop(columns=["id", TARGET], errors="ignore")
y = train_fe[TARGET]
X_test = test_fe.drop(columns=["id"], errors="ignore")

## 4. Modelin Overfit Etmemesi İçin Aldığımız Önlemler
Kaggle'da sızıntı (leakage) yapan veya veriyi ezberleyen modeller genelde "Private Leaderboard" açıklandığında çok düşük skorlar verir. Bunun önüne geçmek için iki mekanizma kurduk:
1. **Sample Weights:** Basit bir CatBoost kurarak tahmini en zor, en gürültülü %5'lik satırları buldum ve modelin bunları ezberlememesi için eğitim ağırlıklarını 0.3'e düşürdüm.
2. **5-Fold Optuna:** Parametreleri ezbere seçmek yerine Optuna'ya bıraktık. Hata yapmamak için Optuna'yı sadece bir fold üzerinde değil, verinin tamamında (5-Fold Stratified) test ederek çalıştırdık.

In [ ]:
y_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
splits = list(skf.split(X, y_bins))

oof_base = np.zeros(len(X))
for tr_i, val_i in splits:
    m = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6,
                          verbose=0, cat_features=cat_cols, random_seed=42)
    m.fit(X.iloc[tr_i], y.iloc[tr_i])
    oof_base[val_i] = m.predict(X.iloc[val_i])

errors = np.abs(y - oof_base)
threshold = np.percentile(errors, 95)
sample_weights = np.where(errors > threshold, 0.3, 1.0)
print(f"Outlier threshold: {threshold:.4f}")

def objective(trial):
    params = {
        "iterations": 2000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 15),
        "eval_metric": "RMSE",
        "verbose": 0,
        "early_stopping_rounds": 100,
        "cat_features": cat_cols,
        "random_seed": 42
    }
    scores = []
    for tr_idx, val_idx in splits:
        model = CatBoostRegressor(**params)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx],
                  sample_weight=sample_weights[tr_idx],
                  eval_set=(X.iloc[val_idx], y.iloc[val_idx]),
                  use_best_model=True)
        pred = model.predict(X.iloc[val_idx])
        scores.append(root_mean_squared_error(y.iloc[val_idx], pred))
    return np.mean(scores)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15, show_progress_bar=True)
best_params = study.best_params
print(f"Best params: {best_params}")

## 5. Kullandığımız Modeller: CatBoost, LightGBM ve Scipy Blend
Tek bir algoritmaya güvenmek yerine, farklı mimarilere sahip CatBoost ve LightGBM'i bir araya getirdim. Varyansı düşürmek için her iki modeli de farklı rastgelelik seedlerle eğittik.

**Final Adımları:**
Modelleri birleştirirken tahminleri "yarı yarıya" almak yerine işi matematiğe bıraktım. **Scipy Minimize** fonksiyonu ile Out-of-Fold tahminlerini analiz ederek en ideal birleştirme yani blend ağırlıklarını buldum.
En son adımda ise, verinin sentetik bir doğası olduğu için, modellerin ürettiği uzun ondalıklı sonuçları gerçeğe en uygun format olan 2 ondalık basamağa yuvarlayarak teslim dosyamı hazırladım.

In [ ]:
SEEDS = [42, 43, 44]

cb_params = {**best_params,
             "iterations": 5000,
             "eval_metric": "RMSE",
             "verbose": 0,
             "early_stopping_rounds": 300,
             "cat_features": cat_cols}

cb_oof = np.zeros(len(X))
cb_test_preds = []

for seed in SEEDS:
    seed_test = []
    for tr_idx, val_idx in splits:
        model = CatBoostRegressor(**cb_params, random_seed=seed)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx],
                  sample_weight=sample_weights[tr_idx],
                  eval_set=(X.iloc[val_idx], y.iloc[val_idx]),
                  use_best_model=True)
        if seed == 42:
            cb_oof[val_idx] = model.predict(X.iloc[val_idx])
        seed_test.append(model.predict(X_test))
    cb_test_preds.append(np.mean(seed_test, axis=0))
cb_final = np.clip(np.mean(cb_test_preds, axis=0), 0, 10)

X_lgb = X.copy()
X_test_lgb = X_test.copy()
for col in cat_cols:
    X_lgb[col] = X_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

lgb_oof = np.zeros(len(X))
lgb_test_preds = []

for seed in SEEDS:
    seed_test = []
    for tr_idx, val_idx in splits:
        model = LGBMRegressor(
            n_estimators=5000,
            learning_rate=0.02,
            num_leaves=31,
            max_depth=6,
            min_child_samples=20,
            random_state=seed,
            verbosity=-1
        )
        model.fit(X_lgb.iloc[tr_idx], y.iloc[tr_idx],
                  sample_weight=sample_weights[tr_idx],
                  eval_set=[(X_lgb.iloc[val_idx], y.iloc[val_idx])],
                  callbacks=[lgb_early_stopping(300, verbose=False)])
        if seed == 42:
            lgb_oof[val_idx] = model.predict(X_lgb.iloc[val_idx])
        seed_test.append(model.predict(X_test_lgb))
    lgb_test_preds.append(np.mean(seed_test, axis=0))
lgb_final = np.clip(np.mean(lgb_test_preds, axis=0), 0, 10)

def blend_loss(weights):
    return root_mean_squared_error(y, weights[0] * cb_oof + weights[1] * lgb_oof)

result = minimize(blend_loss, x0=[0.5, 0.5], method="SLSQP",
                  bounds=[(0,1), (0,1)],
                  constraints={'type':'eq','fun':lambda w: sum(w)-1})
cb_w, lgb_w = result.x
print(f"Optimum Ağırlıklar : CatBoost: %{cb_w*100:.1f} | LightGBM: %{lgb_w*100:.1f}")

final_pred = np.clip(cb_final * cb_w + lgb_final * lgb_w, 0, 10)
print(f"Blend OOF RMSE: {result.fun:.5f}")


In [ ]:
submission = pd.DataFrame({"id": test_ids, TARGET: final_pred})
submission.to_csv("submission_final_stable_v3.csv", index=False)
print("\nsubmission_finalll.csv hazir.")